# Tugas Mandiri — Pertemuan 9: Masalah Sinkronisasi II (Classical Problems)

**Mata Kuliah:** Pemrosesan Paralel dengan Python (2 SKS)  
**Dosen:** Dody Ichwana Putra, Ph.D. (Eng)  
**Program Studi:** Dept. Teknik Komputer  
**Tahun Akademik:** 2026/2027  

---

## 1. Dining Philosophers dengan $N$ Berbeda (Solusi Arbitrator)

Mengulang solusi arbitrator (*waiter* berbasis `Semaphore(N - 1)`) dengan $N = 3$, $N = 7$, dan $N = 10$ filsuf untuk mengukur *throughput* (total aktivitas makan per detik).

In [1]:
import threading
import time

def simulasi_dining_arbitrator(N, putaran=30):
    forks = [threading.Lock() for _ in range(N)]
    kursi_meja = threading.Semaphore(N - 1)
    selesai_makan = [0] * N

    def filsuf(i):
        kiri = forks[i]
        kanan = forks[(i + 1) % N]
        for _ in range(putaran):
            with kursi_meja:
                kiri.acquire()
                kanan.acquire()
                selesai_makan[i] += 1
                time.sleep(0.02)  # simulasi waktu makan
                kanan.release()
                kiri.release()
            time.sleep(0.01)      # simulasi waktu berpikir

    t0 = time.perf_counter()
    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    durasi = time.perf_counter() - t0
    total_makan = sum(selesai_makan)
    throughput = total_makan / durasi
    return total_makan, durasi, throughput

print(f"{'N':<5} | {'Total Makan':<12} | {'Durasi (s)':<12} | {'Throughput (makan/s)':<20}")
print("-" * 58)
for n in [3, 7, 10]:
    total, durasi, tp = simulasi_dining_arbitrator(n, putaran=30)
    print(f"{n:<5} | {total:<12} | {durasi:<12.4f} | {tp:<20.2f}")

N     | Total Makan  | Durasi (s)   | Throughput (makan/s)
----------------------------------------------------------
3     | 90           | 1.8874       | 47.69               
7     | 210          | 1.4825       | 141.65              
10    | 300          | 1.2715       | 235.94              


### Analisis Hasil Soal 1
**Apakah throughput naik linear terhadap $N$?**  
**Ya, throughput meningkat secara proporsional (mendekati linear) terhadap $N$**, namun tetap dibatasi oleh kapasitas pasangan garpu pada topologi meja bundar.

**Penjelasan:**
1. **Kapasitas Paralel Maksimum ($\lfloor N/2 \rfloor$):** Setiap filsuf membutuhkan **2 garpu** yang bersebelahan untuk makan secara bersamaan. Dengan $N$ garpu di meja bundar, maksimal hanya $\lfloor N/2 \rfloor$ filsuf yang dapat makan secara paralel pada saat yang sama ($1$ untuk $N=3$, $3$ untuk $N=7$, dan $5$ untuk $N=10$).
2. **Skala Proporsional:** Karena jumlah maksimum pasangan garpu yang dapat digunakan bersamaan bertambah sebanding dengan $N/2$, kapasitas pemrosesan paralel total ikut naik secara linear seiring bertambahnya jumlah filsuf (dengan sedikit variasi akibat *context switching* thread dan antrean pada semaphore arbitrator `N-1`).

## 2. Readers-Writers dengan Prioritas Writer (*Second Readers-Writers Problem*)

Memodifikasi solusi agar secara eksplisit **memprioritaskan writer**:
- Begitu ada satu atau lebih writer yang menunggu (`write_count > 0`), gerbang `read_try` dikunci sehingga **tidak ada reader baru** yang boleh masuk.
- Reader yang sedang membaca tetap diizinkan menyelesaikan prosesnya hingga `read_count == 0`, barulah `resource` (kunci data) dilepas untuk writer.
- Pengujian dilakukan dengan membanjiri sistem menggunakan thread writer yang datang terus-menerus untuk membuktikan terjadinya **reader starvation**.

In [2]:
import threading
import time

read_count = 0
write_count = 0

mutex_r = threading.Lock()   # melindungi read_count
mutex_w = threading.Lock()   # melindungi write_count
read_try = threading.Lock()  # gerbang masuk reader (dikunci jika ada writer menunggu)
resource = threading.Lock()  # akses eksklusif ke data bersama

berhenti = threading.Event()
total_tulis = [0]
reader_berhasil = threading.Event()

def writer_terus_menerus():
    global write_count
    while not berhenti.is_set():
        with mutex_w:
            write_count += 1
            if write_count == 1:
                read_try.acquire()  # writer PERTAMA menutup gerbang untuk reader baru
        with resource:
            total_tulis[0] += 1
            time.sleep(0.01)        # simulasi menulis data
        with mutex_w:
            write_count -= 1
            if write_count == 0:
                read_try.release()  # writer TERAKHIR membuka kembali gerbang reader

def satu_reader_baru():
    global read_count
    t0 = time.perf_counter()
    with read_try:                  # reader baru tertahan di sini selama write_count > 0
        with mutex_r:
            read_count += 1
            if read_count == 1:
                resource.acquire()
    reader_berhasil.set()
    if not berhenti.is_set():
        print(f"Reader baru berhasil membaca setelah {time.perf_counter()-t0:.2f} detik")
    time.sleep(0.005)
    with mutex_r:
        read_count -= 1
        if read_count == 0:
            resource.release()

writers = [threading.Thread(target=writer_terus_menerus) for _ in range(3)]
for w in writers:
    w.start()

time.sleep(0.1)  # biarkan antrean writer terbentuk terlebih dahulu

reader = threading.Thread(target=satu_reader_baru)
reader.start()
reader.join(timeout=2.0)

if not reader_berhasil.is_set():
    print("Reader baru BELUM mendapat giliran setelah 2.00 detik -- READER STARVATION TERBUKTI!")

berhenti.set()
for w in writers:
    w.join()
reader.join()
print(f"Total operasi tulis yang terjadi selama reader menunggu: {total_tulis[0]}")

Reader baru BELUM mendapat giliran setelah 2.00 detik -- READER STARVATION TERBUKTI!
Total operasi tulis yang terjadi selama reader menunggu: 205


### Analisis Hasil Soal 2
- **Mekanisme Prioritas Writer:** Variabel `write_count` melacak jumlah writer yang sedang aktif maupun menunggu. Writer pertama yang datang segera mengunci `read_try`. Selama masih ada writer lain yang masuk sebelum `write_count` turun ke `0`, `read_try` tetap terkunci.
- **Bukti Reader Starvation:** Pada pengujian di atas, ketika 3 writer terus-menerus datang tanpa jeda, nilai `write_count` tidak pernah mencapai `0`. Akibatnya, thread `satu_reader_baru` tertahan di `with read_try:` dan tidak kunjung mendapat giliran dalam batas waktu 2 detik (*starvation* berpindah dari writer ke reader).

## 3. Sleeping Barber dengan Banyak Tukang Cukur (2 Tukang Cukur)

Memodifikasi simulasi *Sleeping Barber* menjadi **2 tukang cukur** yang berbagi **1 ruang tunggu** berkapasitas 3 kursi.  
Untuk menentukan tukang cukur mana yang melayani pelanggan berikutnya secara teratur, digunakan `queue.Queue()` (`antrian_tukang_cukur`) sebagai jalur koordinasi antara pelanggan yang duduk dan tukang cukur yang siap melayani.

In [3]:
import threading
import queue
import time
import random

KURSI_TUNGGU = 3
JUMLAH_TUKANG_CUKUR = 2

pelanggan_menunggu = threading.Semaphore(0)
mutex = threading.Lock()
jumlah_menunggu = [0]

# Queue menyimpan tuple (id_tukang_cukur, event_selesai_cukur) dari tukang cukur yang siap
antrian_tukang_cukur = queue.Queue()

log_lock = threading.Lock()
def catat(msg):
    with log_lock:
        print(msg)

def tukang_cukur(id_barber):
    while True:
        catat(f"[Tukang Cukur-{id_barber}] tidur, menunggu pelanggan...")
        pelanggan_menunggu.acquire()  # bangun jika ada pelanggan

        with mutex:
            jumlah_menunggu[0] -= 1

        event_selesai = threading.Event()
        # Daftarkan diri ke antrean bahwa tukang cukur ini siap melayani
        antrian_tukang_cukur.put((id_barber, event_selesai))

        catat(f"[Tukang Cukur-{id_barber}] mulai mencukur pelanggan...")
        time.sleep(0.20)  # simulasi durasi potong rambut
        catat(f"[Tukang Cukur-{id_barber}] selesai mencukur!")
        event_selesai.set()  # beri tahu pelanggan spesifik yang sedang dicukur

def pelanggan(id_pelanggan):
    with mutex:
        if jumlah_menunggu[0] < KURSI_TUNGGU:
            jumlah_menunggu[0] += 1
            catat(f"[Pelanggan-{id_pelanggan}] duduk di ruang tunggu ({jumlah_menunggu[0]}/{KURSI_TUNGGU})")
            dapat_kursi = True
        else:
            catat(f"[Pelanggan-{id_pelanggan}] ruang tunggu PENUH, pergi!")
            dapat_kursi = False

    if dapat_kursi:
        pelanggan_menunggu.release()  # bangunkan salah satu tukang cukur
        id_barber, event_selesai = antrian_tukang_cukur.get()  # ambil giliran tukang cukur yang siap
        catat(f"  -> [Pelanggan-{id_pelanggan}] dilayani oleh Tukang Cukur-{id_barber}")
        event_selesai.wait()  # tunggu hingga Tukang Cukur tersebut selesai

# Jalankan 2 thread tukang cukur
for b in range(JUMLAH_TUKANG_CUKUR):
    threading.Thread(target=tukang_cukur, args=(b,), daemon=True).start()

time.sleep(0.05)
random.seed(42)
pelanggan_threads = []
for i in range(10):
    time.sleep(random.uniform(0.02, 0.08))
    t = threading.Thread(target=pelanggan, args=(i,))
    t.start()
    pelanggan_threads.append(t)

for t in pelanggan_threads:
    t.join()

print("\nSimulasi 2 tukang cukur selesai.")

[Tukang Cukur-0] tidur, menunggu pelanggan...
[Tukang Cukur-1] tidur, menunggu pelanggan...
[Pelanggan-0] duduk di ruang tunggu (1/3)
[Tukang Cukur-0] mulai mencukur pelanggan...
  -> [Pelanggan-0] dilayani oleh Tukang Cukur-0
[Pelanggan-1] duduk di ruang tunggu (1/3)
[Tukang Cukur-1] mulai mencukur pelanggan...
  -> [Pelanggan-1] dilayani oleh Tukang Cukur-1
[Pelanggan-2] duduk di ruang tunggu (1/3)
[Pelanggan-3] duduk di ruang tunggu (2/3)
[Pelanggan-4] duduk di ruang tunggu (3/3)
[Tukang Cukur-0] selesai mencukur!
[Tukang Cukur-0] tidur, menunggu pelanggan...
[Tukang Cukur-0] mulai mencukur pelanggan...
  -> [Pelanggan-2] dilayani oleh Tukang Cukur-0
[Pelanggan-5] duduk di ruang tunggu (3/3)
[Tukang Cukur-1] selesai mencukur!
[Tukang Cukur-1] tidur, menunggu pelanggan...
[Tukang Cukur-1] mulai mencukur pelanggan...
  -> [Pelanggan-3] dilayani oleh Tukang Cukur-1
[Pelanggan-6] duduk di ruang tunggu (3/3)
[Pelanggan-7] ruang tunggu PENUH, pergi!
[Pelanggan-8] ruang tunggu PENUH, pergi

### Analisis Hasil Soal 3
- **Koordinasi Multi-Barber:** Dengan menggunakan `queue.Queue()`, setiap tukang cukur yang terbangun dan mengurangi `jumlah_menunggu` mengirimkan identitasnya beserta objek sinkronisasi personal (`threading.Event`).
- Pelanggan mengambil pasangan `(id_barber, event_selesai)` secara FIFO dari `antrian_tukang_cukur`, sehingga pasangan tukang cukur dan pelanggan terpetakan secara akurat tanpa tertukar sinyal meskipun kedua tukang cukur bekerja bersamaan.

## 4. Refleksi Singkat

Menurut saya, **The Sleeping Barber Problem** adalah masalah klasik yang paling mirip dengan skenario nyata dalam pengembangan perangkat lunak modern, khususnya pada arsitektur **Web Server / Thread Pool dan Database Connection Pool**.

**Contoh Konkret:**  
Pada server web (seperti NGINX atau *worker pool* pada API backend), terdapat jumlah *worker thread* terbatas (analog dengan **tukang cukur**) dan antrean permintaan masuk (*request queue* / *backlog socket*) berkapasitas tetap (analog dengan **kursi tunggu**). Ketika tidak ada *request* masuk, *worker thread* masuk ke mode *idle/sleep* agar tidak membebani CPU. Saat lonjakan trafik (*traffic spike*) terjadi hingga seluruh *worker* sibuk dan kapasitas antrean penuh, server langsung menerapkan mekanisme *load shedding* dengan menolak *request* baru menggunakan kode status HTTP `503 Service Unavailable` atau `429 Too Many Requests` (analog dengan **pelanggan yang langsung pergi**), sehingga server terhindar dari kehabisan memori (*Out-of-Memory*).